# Anatomía de moondream2: de la imagen a la respuesta, paso a paso

Este notebook **no evalúa nada**: abre el modelo y ejecuta a mano cada paso interno, mostrando qué entra, qué sale y con qué forma. En cada paso **compara el resultado manual con el del propio modelo**. Si coinciden, queda demostrado que eso es exactamente lo que el modelo hace por dentro.

**Cómo usarlo**
1. Colab con GPU (T4 basta). Sube **una imagen** del benchmark sintético (por ejemplo `escena_0025_n005.png`) al panel de archivos.
2. Corre la celda de instalación, **reinicia la sesión** y luego corre el resto en orden.

Versión del modelo: `vikhyatk/moondream2`, revisión `2025-06-21` (la misma de tus pilotos).
> Escrito sin poder ejecutarse fuera de Colab. Si algo falla, el error dirá en qué paso; los `assert` están puestos para que falle de forma visible y no en silencio.

In [ ]:
!pip install -q "transformers==4.46.3"
print("Listo. REINICIA la sesión (Entorno de ejecución > Reiniciar sesión) y continúa.")

## 0. ¿Dónde "vive" el modelo?

Un modelo de Hugging Face son **dos cosas descargadas a tu sesión de Colab**:
- **Un archivo de pesos** (`model.safetensors`): millones de números aprendidos durante el entrenamiento.
- **Código Python** (`vision.py`, `text.py`, `layers.py`…). Con `trust_remote_code=True` le das permiso a `transformers` para descargar y ejecutar el código que escribió el autor del modelo.

Todo lo que sigue ocurre en la GPU de Colab, dentro de esas funciones de Python.

In [ ]:
import sys, inspect, torch, numpy as np
import torch.nn.functional as F
from transformers import AutoModelForCausalLM
from PIL import Image
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    "vikhyatk/moondream2", revision="2025-06-21", trust_remote_code=True,
    **({"torch_dtype": torch.float16} if device == "cuda" else {})
).to(device).eval()

md = model.model          # el objeto HF envuelve al modelo real (MoondreamModel) en .model
_ = model.encode_image    # (acceder a esta propiedad prepara la memoria del LLM)

# localizar los módulos de código descargados
mods = {n.split(".")[-1]: m for n, m in sys.modules.items() if m is not None and "moondream" in n.lower()}
vision_mod, layers_mod, crops_mod = mods["vision"], mods["layers"], mods["image_crops"]
print("Código que se está ejecutando (ábrelo con el explorador de archivos de Colab):")
for k in ["moondream", "vision", "layers", "image_crops", "text"]:
    print(f"  {k:12s} -> {inspect.getsourcefile(mods[k])}")

try:
    from huggingface_hub import scan_cache_dir
    for repo in scan_cache_dir().repos:
        if repo.repo_id == "vikhyatk/moondream2":
            for rev in repo.revisions:
                for f in rev.files:
                    if f.file_name.endswith((".safetensors", ".bin")):
                        print(f"Pesos: {f.file_path}  ({f.size_on_disk/1e9:.2f} GB)")
except Exception as e:
    print("(no se pudo listar el caché:", e, ")")

n_vis = sum(p.numel() for p in md.vision.parameters()); n_all = sum(p.numel() for p in md.parameters())
print(f"\nParámetros: total {n_all/1e9:.2f} mil millones | encoder visual {n_vis/1e6:.0f} millones")
print("Algunas matrices del encoder visual (nombre -> forma):")
for name, p in list(md.vision.named_parameters())[:6]:
    print(f"  {name:28s} {tuple(p.shape)}")
cfg = md.config.vision
print("\nConfiguración visual:", cfg)

### El código exacto del encoder

Esta es la función que convierte parches en vectores. La imprimimos desde el archivo descargado, no de memoria:

In [ ]:
print(inspect.getsource(vision_mod.vision_encoder))
print(inspect.getsource(layers_mod.attn))

## 1. La imagen es una tabla de números

Un PNG en disco son bytes comprimidos. Al abrirlo con PIL y pasarlo a NumPy queda una tabla de `alto × ancho × 3` (rojo, verde, azul), con valores entre 0 y 255.

In [ ]:
IMG_PATH = "/content/escena_0025_n005.png"   # <- ajusta al nombre de tu imagen
img = Image.open(IMG_PATH).convert("RGB")
arr = np.asarray(img)
print("forma:", arr.shape, "| tipo:", arr.dtype, "| cantidad de números:", arr.size)
print("pixel (0,0):", arr[0, 0])
plt.imshow(img); plt.axis("off"); plt.show()

## 2. Paso 1: recortes (`overlap_crop_image` / `prepare_crops`)

"Recortar" es literalmente **seleccionar filas y columnas de la tabla** (`arr[y0:y1, x0:x1]`). moondream arma:
- **Recorte 0:** la imagen completa redimensionada a 378×378 (vista global).
- **Recortes 1..N:** la imagen agrandada y cortada en ventanas de 378×378 que se traslapan (vistas de zoom).

In [ ]:
out = crops_mod.overlap_crop_image(arr, max_crops=cfg.max_crops, overlap_margin=cfg.overlap_margin)
crops_np, tiling = out["crops"], out["tiling"]
print("recortes:", crops_np.shape, "| grilla de zoom:", tiling)
fig, axs = plt.subplots(1, len(crops_np), figsize=(2.2 * len(crops_np), 2.4))
for i, ax in enumerate(np.atleast_1d(axs)):
    ax.imshow(crops_np[i]); ax.set_title("global" if i == 0 else f"zoom {i}", fontsize=8); ax.axis("off")
plt.show()

crops_t, tiling_t = vision_mod.prepare_crops(img, cfg, md.device)   # lo mismo, en la versión que usa el modelo
print("tensor que entra al encoder:", tuple(crops_t.shape), crops_t.dtype,
      f"| valores entre {crops_t.min().item():.2f} y {crops_t.max().item():.2f}  (0..255 pasado a -1..1)")
assert tuple(tiling_t) == tuple(tiling)

## 3. Paso 2: parches (`create_patches`)

Cada recorte de 378×378 se corta en 27×27 cuadritos de 14×14 px, y cada cuadrito se aplana a 14·14·3 = 588 números. Esto todavía no es nada aprendido: es reordenar la tabla.

In [ ]:
W_DTYPE = md.vision.patch_emb.weight.dtype
x = crops_t.to(W_DTYPE)
patches = vision_mod.create_patches(x, cfg.enc_patch_size)
print("parches:", tuple(patches.shape), " = [recortes, parches por recorte, números por parche]")

# verificación: el parche 0 del recorte 0 es exactamente el cuadrito superior izquierdo
manual = x[0, :, 0:14, 0:14].permute(1, 2, 0).reshape(-1)   # [588]
print("¿el parche (0,0) coincide con el cuadrito arr[0:14,0:14]?",
      torch.allclose(patches[0, 0].float(), x[0, :, 0:14, 0:14].reshape(-1).float()) or
      torch.allclose(patches[0, 0].float(), manual.float()))

## 4. Paso 3: el encoder visual, ejecutado a mano

Repetimos el contenido de `vision_encoder` línea por línea y lo comparamos con el resultado del modelo:
- `patch_emb`: multiplicación por una matriz aprendida de 588×1152. Cada parche pasa a 1152 números.
- `+ pos_emb`: se suma un vector aprendido por cada posición, para que el modelo sepa *dónde* está cada parche.
- 27 bloques. En cada uno, **atención** (cada parche mezcla información de los demás parches de su recorte) y **MLP** (cada parche se transforma por separado).

In [ ]:
w = md.vision
with torch.no_grad():
    h = w.patch_emb(patches);  print("tras patch_emb :", tuple(h.shape))
    h = h + w.pos_emb;         print("tras pos_emb   :", tuple(h.shape))
    for i, block in enumerate(w.blocks):
        h = h + vision_mod.attn(layers_mod.layer_norm(h, block.ln1), block.attn, n_heads=cfg.enc_n_heads)
        h = h + layers_mod.mlp(layers_mod.layer_norm(h, block.ln2), block.mlp)
        if i in (0, 13, len(w.blocks) - 1):
            print(f"tras bloque {i+1:2d}  :", tuple(h.shape))
    h = layers_mod.layer_norm(h, w.post_ln)
    ref = md._vis_enc(x)       # la función que usa el modelo de verdad
diff = (h.float() - ref.float()).abs().max().item()
print(f"\n¿manual == modelo? diferencia máxima = {diff:.2e}")
assert diff < 1e-2, "no coinciden: revisar versión del modelo"

### ¿Qué aprendió cada vector? Similitud entre parches

Elegimos un parche de una figura en la vista global y medimos cuánto se parece su vector a los otros 728 (similitud coseno). Si el encoder hizo su trabajo, deberían parecerse los parches con contenido similar, no los que tienen el mismo color de píxel por casualidad.

In [ ]:
g = ref[0].float()                                     # vista global: [729, 1152]
gray = (np.abs(np.asarray(img.resize((378, 378))).astype(int) - 200).sum(-1) > 60)
score = gray.reshape(27, 14, 27, 14).mean((1, 3))      # fracción de "figura" en cada parche
r, c = np.unravel_index(score.argmax(), score.shape); k = r * 27 + c
sim = F.cosine_similarity(g[k:k+1], g, dim=-1).reshape(27, 27).cpu().numpy()
fig, axs = plt.subplots(1, 2, figsize=(9, 4.3))
axs[0].imshow(img.resize((378, 378))); axs[0].add_patch(plt.Rectangle((c*14, r*14), 14, 14, fill=False, ec="red", lw=2)); axs[0].axis("off")
axs[0].set_title(f"parche elegido (fila {r}, col {c})")
im = axs[1].imshow(sim, cmap="viridis"); axs[1].set_title("similitud de su vector con los 729"); plt.colorbar(im, ax=axs[1])
plt.show()

## 5. Pasos 4 y 5: fusión global + zoom y traducción al LLM

- Los zooms se pegan en un solo mapa, quitando los bordes traslapados (`reconstruct_from_crops`), y se promedian hasta 27×27.
- Cada casilla une su vector global (1152) con su vector de zoom (1152) = 2304.
- Una pequeña red (`proj_mlp`) lo traduce a 2048, el tamaño de las "palabras" del LLM.

In [ ]:
with torch.no_grad():
    local = ref[1:].view(-1, cfg.enc_n_layers, cfg.enc_n_layers, cfg.enc_dim)
    recon = crops_mod.reconstruct_from_crops(local, tiling_t, patch_size=1, overlap_margin=cfg.overlap_margin)
    print("zoom reensamblado:", tuple(recon.shape))
    pooled = F.adaptive_avg_pool2d(recon.permute(2, 0, 1), (27, 27)).permute(1, 2, 0).reshape(729, cfg.enc_dim)
    final = torch.cat([ref[0], pooled], dim=-1);   print("global + zoom   :", tuple(final.shape))
    tokens_img = layers_mod.mlp(final, w.proj_mlp); print("palabras visuales:", tuple(tokens_img.shape))
    ref2 = md._run_vision_encoder(img)
diff2 = (tokens_img.float() - ref2.float()).abs().max().item()
print(f"¿manual == modelo? diferencia máxima = {diff2:.2e}")

## 6. Paso 6: el LLM lee la imagen y la guarda

`encode_image` antepone el token de inicio (BOS) a las 729 palabras visuales, las pasa por las 24 capas del LLM y guarda el resultado en la *KV-cache*: una memoria con lo que cada capa calculó para cada posición. Fíjate que **no recibe ninguna pregunta**.

In [ ]:
enc = model.encode_image(img)
print("posiciones ocupadas por la imagen:", enc.pos, "(= 1 BOS + 729)")
print("capas del LLM con memoria guardada:", len(enc.caches))
print("forma de la memoria de la capa 1 (claves):", tuple(enc.caches[0][0].shape), "= [lote, cabezas, posiciones, dim]")

## 7. Paso 7: la pregunta entra recién aquí

Reutilizamos **la misma imagen ya codificada** para dos preguntas distintas. El paso 6 no se repite, y eso demuestra que la parte visual no depende de la pregunta.

In [ ]:
print("plantilla de pregunta:", md.config.tokenizer.templates["query"])
for q in ["How many shapes are there in total in the image?", "How many blue shapes are there in the image?"]:
    ids = md.tokenizer.encode(q).ids
    print(f"\nPregunta: {q}\n  tokens de texto ({len(ids)}): {ids}")
    print("  respuesta:", model.query(enc, q)["answer"])